## Step 1: prepare the safety file for Tableau

- Tableau works best with one flat file. I take the California table with the EU flags (california_eu.csv), add a readable EU status label (banned, restricted, not banned) and a readable removal label, and save it to data/tableau/. First I look at the column names, so the file only contains what the dashboard needs.

In [1]:
import pandas as pd

ca = pd.read_csv("../data/clean/california_eu.csv")

# 1. Which columns do we have?
print(list(ca.columns))
ca.head(3)

['CDPHId', 'ProductName', 'CSFId', 'CSF', 'CompanyId', 'CompanyName', 'BrandName', 'PrimaryCategoryId', 'PrimaryCategory', 'SubCategoryId', 'SubCategory', 'CasId', 'CasNumber', 'ChemicalId', 'ChemicalName', 'InitialDateReported', 'MostRecentDateReported', 'DiscontinuedDate', 'ChemicalCreatedAt', 'ChemicalUpdatedAt', 'ChemicalDateRemoved', 'ChemicalCount', 'is_removed', 'is_discontinued', 'is_tio2', 'is_trade_secret', 'eu_banned', 'eu_restricted']


,CDPHId,ProductName,CSFId,CSF,CompanyId,CompanyName,BrandName,PrimaryCategoryId,PrimaryCategory,SubCategoryId,...,ChemicalCreatedAt,ChemicalUpdatedAt,ChemicalDateRemoved,ChemicalCount,is_removed,is_discontinued,is_tio2,is_trade_secret,eu_banned,eu_restricted
0,2,ULTRA COLOR RICH EXTRA PLUMP LIPSTICK-ALL SHADES,NaN,NaN,4,New Avon LLC,AVON,44,Makeup Products (non-permanent),53,...,2009-07-09,2009-07-09,NaN,1,False,True,True,False,False,True
1,3,Glover's Medicated Shampoo,NaN,NaN,338,J. Strickland & Co.,Glover's,18,Hair Care Products (non-coloring),25,...,2009-07-01,2009-07-01,NaN,2,False,False,False,False,False,False
2,3,Glover's Medicated Shampoo,NaN,NaN,338,J. Strickland & Co.,Glover's,18,Hair Care Products (non-coloring),25,...,2009-07-02,2009-07-02,NaN,2,False,False,False,False,False,False


## Step 2: Build the flat file for Tableau
- I add readable labels (EU status, removed or not), the year of the first report, and a flag in_h1_test that marks exactly the reports used in the H1 test (no titanium dioxide, CAS number known), so the dashboard shows the same 16.7% vs 5.9% as my analysis. I keep only the columns the dashboard needs, with Tableau-friendly names.

In [2]:
# 1. Readable EU status (banned wins if both were true; the check earlier showed there is no overlap)
ca["eu_status"] = "Other"                                                    # default: not banned or restricted
ca.loc[ca["eu_restricted"].fillna(False).astype(bool), "eu_status"] = "EU-restricted"
ca.loc[ca["eu_banned"].fillna(False).astype(bool),     "eu_status"] = "EU-banned"

# 2. Readable removal label and year of the first report
ca["removed"] = ca["is_removed"].map({True: "Removed", False: "Still reported"})
ca["first_report_year"] = pd.to_datetime(ca["InitialDateReported"], errors="coerce").dt.year

# 3. Flag for exactly the reports used in the H1 test (no TiO2, CAS number known)
ca["in_h1_test"] = (~ca["is_tio2"].astype(bool)) & ca["CasNumber"].notna()

# 4. Keep the columns the dashboard needs, with simple names
tab = ca[["CDPHId", "ProductName", "BrandName", "CompanyName", "PrimaryCategory",
          "ChemicalName", "CasNumber", "eu_status", "removed", "first_report_year",
          "is_tio2", "in_h1_test"]].rename(columns={
          "CDPHId": "product_id", "ProductName": "product_name", "BrandName": "brand",
          "CompanyName": "company", "PrimaryCategory": "category",
          "ChemicalName": "chemical", "CasNumber": "cas_number"})

# 5. Checks: must match the earlier results
print(tab["eu_status"].value_counts())
h1 = tab[tab["in_h1_test"]]
print("\nremoval share in the H1 base (should be about 16.7% banned vs 5.9% other):")
print(h1.groupby("eu_status")["removed"].apply(lambda s: (s == "Removed").mean() * 100).round(1))

# 6. Save for Tableau
tab.to_csv("../data/tableau/safety_reports.csv", index=False)
print("\nsaved", len(tab), "rows")

eu_status
EU-restricted    97677
Other            15859
EU-banned          845
Name: count, dtype: int64

removal share in the H1 base (should be about 16.7% banned vs 5.9% other):
eu_status
EU-banned        16.7
EU-restricted     3.0
Other             7.4
Name: removed, dtype: float64

saved 114381 rows


In [3]:
# Two groups exactly as in the H1 test: banned vs everything else
tab["banned_group"] = tab["eu_status"].map(lambda s: "EU-banned" if s == "EU-banned" else "Not banned")

h1 = tab[tab["in_h1_test"]]
print(h1.groupby("banned_group")["removed"].apply(lambda s: (s == "Removed").mean() * 100).round(1))   # should be 16.7 and 5.9

tab.to_csv("../data/tableau/safety_reports.csv", index=False)
print("saved", len(tab), "rows")

banned_group
EU-banned     16.7
Not banned     5.9
Name: removed, dtype: float64
saved 114381 rows


In [4]:
b = ca[ca["eu_banned"].fillna(False).astype(bool)]            # the 845 banned reports

print("chemicals by name:", b["ChemicalName"].nunique(), "| by CAS number:", b["CasNumber"].nunique())
print("companies by name:", b["CompanyName"].nunique(), "| by company ID:", b["CompanyId"].nunique())

# Which chemical names share a CAS number?
dup = b.groupby("CasNumber")["ChemicalName"].nunique()
print(b[b["CasNumber"].isin(dup[dup > 1].index)][["CasNumber", "ChemicalName"]].drop_duplicates())

# Which company names have more than one ID?
dupc = b.groupby("CompanyName")["CompanyId"].nunique()
print(dupc[dupc > 1])

chemicals by name: 38 | by CAS number: 37
companies by name: 84 | by company ID: 86
       CasNumber      ChemicalName
27825  8007-45-2          Coal Tar
39366  8007-45-2  Coal tar extract
CompanyName
American International Industries    2
Regis Corporation                    2
Name: CompanyId, dtype: int64


**Findings: counting banned chemicals and companies**
- Banned chemicals: 37 by CAS number (38 by name, because "Coal Tar" and "Coal tar extract" share CAS 8007-45-2). The CAS number is used as the unit.
- Companies with a banned chemical: 84 by name (86 by ID; American International Industries and Regis Corporation each have two IDs). The name is used as the unit.
- Reports 845 and products 455 are unaffected.

In [5]:
# Who reported the banned chemicals first in 2016?
y16 = tab[(tab["eu_status"] == "EU-banned") & (tab["first_report_year"] == 2016)]
print(y16["company"].value_counts().head(5))
print(y16["chemical"].value_counts().head(5))

company
La Prairie, Inc.                        188
Johnson & Johnson Consumer Companies      3
Greenbrier International, Inc.            1
Name: count, dtype: int64
chemical
Benzophenone      180
Coal Tar            3
Safrole             2
Acetaldehyde        2
Benzo[a]pyrene      2
Name: count, dtype: int64


- Robustness check: La Prairie reported 188 of the 845 banned reports in 2016 (mostly benzophenone). I compare the removal rates with and without this company, to see whether the H1 result depends on one company.

In [6]:
# H1 robustness: the removal rates without La Prairie
h1 = tab[tab["in_h1_test"]]
for label, d in [("all companies", h1), ("without La Prairie", h1[h1["company"] != "La Prairie, Inc."])]:
    r = d.groupby("banned_group")["removed"].apply(lambda s: (s == "Removed").mean() * 100).round(1)
    n = d.groupby("banned_group").size()
    print(label, "->", r.to_dict(), "| reports:", n.to_dict())

all companies -> {'EU-banned': 16.7, 'Not banned': 5.9} | reports: {'EU-banned': 845, 'Not banned': 13338}
without La Prairie -> {'EU-banned': 19.2, 'Not banned': 5.9} | reports: {'EU-banned': 657, 'Not banned': 13334}


**Findings: robustness of H1 without La Prairie**
- La Prairie, Inc. reported 188 of the 845 EU-banned reports (22%), mostly benzophenone (180), as a batch in 2016.
- Removal rate of EU-banned chemicals: 16.7% with all companies, 19.2% without La Prairie; not banned: 5.9% in both.
- La Prairie's banned reports were rarely removed (roughly 8%), so including them lowers the banned removal rate. The H1 result does not depend on this one company; it is slightly conservative.

In [7]:
b = tab[tab["eu_status"] == "EU-banned"]
for label, d in [("all companies", b), ("without La Prairie", b[b["company"] != "La Prairie, Inc."])]:
    print(label)
    print(d.groupby("category")["product_id"].nunique().sort_values(ascending=False).head(4), "\n")

all companies
category
Skin Care Products                   119
Hair Care Products (non-coloring)     97
Makeup Products (non-permanent)       66
Nail Products                         54
Name: product_id, dtype: int64 

without La Prairie
category
Hair Care Products (non-coloring)    97
Skin Care Products                   70
Nail Products                        54
Makeup Products (non-permanent)      51
Name: product_id, dtype: int64 

